In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [ ]:
ACCESS_THRESHOLD = -0.2
df = pd.read_csv("/content/Копия Badtrain  - Лист1.csv")
texts = df["Решение кейса"].fillna("").astype(str)
y = df["Допуск"].astype(int).values
groups = df["Кейс"].values

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                              min_df=1, sublinear_tf=True)),
    ("svm", LinearSVC(C=1.0, max_iter=5000, random_state=42)),
])
cv = GroupKFold(n_splits=5)
val_scores = cross_val_predict(
    model, texts, y, groups=groups, cv=cv, method="decision_function")

In [ ]:
thresholds = np.round(np.arange(-1.0, 1.01, 0.1), 2)
rows = []
for t in thresholds:
    preds = (val_scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, preds, labels=[0, 1]).ravel()
    rows.append({
        "threshold": t,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "recall": round(tp / (tp + fn), 3),
        "precision": round(tp / max(tp + fp, 1), 3),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

 threshold  TP  TN  FP  FN  recall  precision
      -1.0 102   0  98   0   1.000      0.510
      -0.9 102   2  96   0   1.000      0.515
      -0.8 102  12  86   0   1.000      0.543
      -0.7 101  20  78   1   0.990      0.564
      -0.6 101  28  70   1   0.990      0.591
      -0.5 101  41  57   1   0.990      0.639
      -0.4 101  57  41   1   0.990      0.711
      -0.3 101  70  28   1   0.990      0.783
      -0.2 101  81  17   1   0.990      0.856
      -0.1  99  85  13   3   0.971      0.884
      -0.0  96  90   8   6   0.941      0.923
       0.1  95  94   4   7   0.931      0.960
       0.2  91  95   3  11   0.892      0.968
       0.3  88  97   1  14   0.863      0.989
       0.4  79  97   1  23   0.775      0.988
       0.5  69  97   1  33   0.676      0.986
       0.6  58  97   1  44   0.569      0.983
       0.7  41  98   0  61   0.402      1.000
       0.8  24  98   0  78   0.235      1.000
       0.9  16  98   0  86   0.157      1.000
       1.0   6  98   0  96   0.059

In [ ]:
preds = (val_scores >= ACCESS_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y, preds, labels=[0, 1]).ravel()
print(f"TP {tp}")
print(f"TN {tn}")
print(f"FP {fp}")
print(f"FN {fn}")
print()
print(classification_report(
    y, preds, target_names=["не допуск", "допуск"]))

TP 101
TN 70
FP 28
FN 1

              precision    recall  f1-score   support

   не допуск       0.99      0.71      0.83        98
      допуск       0.78      0.99      0.87       102

    accuracy                           0.85       200
   macro avg       0.88      0.85      0.85       200
weighted avg       0.88      0.85      0.85       200



In [ ]:
model.fit(texts, y)
joblib.dump(model, "model_decision_tfidf_svm.joblib")

['model_decision_tfidf_svm.joblib']

In [ ]:
model = joblib.load("model_decision_tfidf_svm.joblib")
# Функция допуска решения
def predict_decision(text):
  # возращает 1, если решение допускается, возращает 0 если нет
    score = model.decision_function([str(text)])[0]
    return int(score >= ACCESS_THRESHOLD)

In [ ]:
text = (
        "Бизнес это очень важно. Нужно делать хорошо и не делать плохо. "
        "Клиенты любят когда им хорошо. Банк должен быть хорошим и "
        "помогать людям. Всё будет отлично потому что так надо. "
        "Главное верить в успех и тогда всё получится. Деньги это тоже "
        "важно, их нужно много зарабатывать."
)
text_access = (
    "Я выбрал отрасль частных детских садов и центров раннего "
    "развития. Рынок растет на 15% в год (данные BusinesStat), "
    "количество частных садов — 8000, 60% — МСБ. ЦА разделена на три "
    "сегмента. Первый — микро-сады на дому (до 10 детей), проблема: "
    "родители платят в начале месяца, но если ребёнок заболел и не "
    "ходит 2 недели, требуют перерасчет, сад теряет до 20% выручки. "
    "Второй — малые сады с группами до 30 детей и лицензией, "
    "проблема: аренда и зарплата — 70% расходов, а сезонные колебания "
    "(летом детей меньше) создают кассовые разрывы. Третий — сети из "
    "3–5 садов, проблема: управление ликвидностью между точками и "
    "длинные деньги от муниципальных контрактов на компенсацию "
    "родительской платы (оплата через 60 дней). Я использовал данные "
    "Росстата и провел 6 интервью с владельцами садов. Решение — "
    "экосистема «Альфа.Дети»: страховка от невыхода ребёнка (партнёр "
    "«АльфаСтрахование») с выплатой саду 80% оплаченного дня, "
    "автоматический расчет по QR-кодам входа-выхода, кредитная линия "
    "под муниципальные контракты с льготным периодом до получения "
    "оплаты, интеграция с детскими CRM (партнёр «АйТиДети») для "
    "автоматической рассылки квитанций, касса для приёма платежей с "
    "рассрочкой через партнёра «Долями». Отличие от конкурентов: ни "
    "Сбер, ни Т-Банк не предлагают страховку от невыхода для детских "
    "садов и интеграцию с муниципальными контрактами. По финансам: "
    "разработка — 8 млн, интеграции — 3 млн, маркетинг — 2 млн. "
    "Прогноз: 200 клиентов в первый год, средний доход — 30 тыс. "
    "рублей (страховая премия 8% от абонемента + комиссия за кредит "
    "2%). Выручка — 6 млн. Окупаемость — 2,2 года. Риски: "
    "регуляторный (20%) — решение: юридический мониторинг и быстрая "
    "адаптация, страховой мошенничества (15%) — требование справки от "
    "педиатра, конкурентный (60%) — эксклюзив с топ-3 CRM. "
    "Доказательства: 6 интервью, данные Росстата, пилот на 5 садах "
    "показал снижение потерь от невыходов с 20% до 4%. Финансовая "
    "модель слабее — нет CAC, LTV, NPV, трёх сценариев, только "
    "базовый расчет."
)
print(predict_decision(text))

0
